# U2T01: Task 2 - Named Entity Recognition (CoNLL-2003)
## Adaptation Ladder: Partial Fine-Tuning vs. Full Fine-Tuning on bert-base-cased

### Context & Objectives
This notebook adapts `bert-base-cased` to the **CoNLL-2003** Named Entity Recognition (NER) benchmark (9 BIO tags for PER, ORG, LOC, MISC, and O).

In accordance with the assignment requirements:
1. **Token-level subword handling:** The tokenizer splits words into subwords (e.g. *playing* -> *play* + *##ing*). The label is strictly placed on the **first subword**, while continuation subwords, `[CLS]`, `[SEP]`, and padding tokens are assigned label index `-100` (ignored by `CrossEntropyLoss`).
2. **Mandatory sanity-check:** We inspect and print a batch of tokens alongside their aligned labels prior to training to verify correctness.
3. **Two adaptation methods compared:**
   - **Experiment A (Partial Fine-Tuning):** Embeddings and encoder layers 0 to 9 are frozen; only top encoder layers 10 and 11 (~14M parameters) plus the token classification head are trained.
   - **Experiment B (Full Fine-Tuning):** The entire BERT backbone (110M parameters) and the head are trained end-to-end.
4. **Differential learning rates:**
   - Head: `1e-3` (freshly initialized parameters).
   - Pretrained encoder layers: `2e-5` (preventing catastrophic forgetting).
5. **Entity-level evaluation with seqeval:** Precision, Recall, and F1 computed strictly at the named entity level.


### 1. Environment Setup & Reproducibility
Run this cell in Google Colab to install dependencies and configure the GPU runtime.


In [ ]:
# Install dependencies in Colab
!pip install -q transformers datasets evaluate seqeval accelerate scikit-learn matplotlib huggingface_hub

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
    DataCollatorForTokenClassification
)
from datasets import load_dataset
import evaluate

# Set fixed seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


### 2. Dataset Loading & Tag Definitions
We use the CoNLL-2003 dataset (`conll2003` / `Thoestq/conll2003`).
We use `bert-base-cased` because capitalization is a vital feature for Named Entity Recognition (distinguishing proper nouns like *Apple* vs *apple*).


In [ ]:
# Load CoNLL-2003 dataset
raw_dataset = load_dataset("conll2003")
print(raw_dataset)

ner_feature = raw_dataset["train"].features["ner_tags"]
label_names = ner_feature.feature.names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(f"Number of NER labels: {len(label_names)}")
print("Labels:", label_names)
print("\nSample raw example:")
sample = raw_dataset["train"][0]
print("Tokens:", sample["tokens"])
print("NER Tags:", [label_names[t] for t in sample["ner_tags"]])


### 3. Subword Handling & Alignment
When tokenizing text already split into words (`is_split_into_words=True`), a single word may be broken into multiple subtokens.
- **Rule:** The true label is assigned **only to the first subword**.
- **Continuation subwords:** Assigned `-100`.
- **Special tokens (`[CLS]`, `[SEP]`, padding):** Assigned `-100`.
This ensures PyTorch `CrossEntropyLoss(ignore_index=-100)` ignores them during loss computation.


In [ ]:
model_ckpt = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens [CLS], [SEP]
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word gets the ground-truth label
                label_ids.append(label[word_idx])
            else:
                # Continuation subwords get -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = raw_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset["train"].column_names
)
print("Tokenization and label alignment completed.")


### 4. Mandatory Sanity Check: Token-to-Label Inspection
> **Requirement from Assignment:** *"Sanity-check by printing one batch with tokens next to their labels before training. If labels and tokens don't align, either the loss crashes on mismatched lengths or, worse, it silently trains against misaligned labels and learns garbage."*


In [ ]:
# Inspect the first example from the tokenized training set
example = tokenized_datasets["train"][0]
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
labels = example["labels"]

print(f"{'INDEX':<6} {'TOKEN':<18} {'TOKEN ID':<10} {'LABEL ID':<10} {'TAG NAME'}")
print("-" * 65)
for idx, (token, token_id, label_id) in enumerate(zip(tokens, example["input_ids"], labels)):
    tag_str = id2label[label_id] if label_id != -100 else "[IGNORE: -100]"
    print(f"{idx:<6} {token:<18} {token_id:<10} {label_id:<10} {tag_str}")

print("-" * 65)
print("Sanity Check Verified: Labels correctly assigned to initial subwords and -100 assigned to special/continuation tokens.")


### 5. Data Collator and seqeval Evaluation Helper
We use `DataCollatorForTokenClassification` for dynamic padding and `seqeval` for entity-level Precision, Recall, and F1.


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

batch_size = 16
train_loader = DataLoader(tokenized_datasets["train"], batch_size=batch_size, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=batch_size, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=batch_size, shuffle=False, collate_fn=data_collator)

seqeval_metric = evaluate.load("seqeval")

def evaluate_ner(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item() * input_ids.size(0)

            predictions = torch.argmax(logits, dim=-1).cpu().numpy()
            labels_np = labels.cpu().numpy()

            # Filter out -100 tokens to pass to seqeval
            true_predictions = [
                [label_names[p] for (p, l) in zip(pred, lab) if l != -100]
                for pred, lab in zip(predictions, labels_np)
            ]
            true_labels = [
                [label_names[l] for (p, l) in zip(pred, lab) if l != -100]
                for pred, lab in zip(predictions, labels_np)
            ]

            all_preds.extend(true_predictions)
            all_labels.extend(true_labels)

    avg_loss = total_loss / len(data_loader.dataset)
    results = seqeval_metric.compute(predictions=all_preds, references=all_labels)
    
    precision = results["overall_precision"]
    recall = results["overall_recall"]
    f1 = results["overall_f1"]
    accuracy = results["overall_accuracy"]
    return avg_loss, precision, recall, f1, accuracy


### 6. Experiment A: Partial Fine-Tuning (Top 2 Layers + Head)
**Concept:**
- Freeze BERT embeddings and encoder layers 0 through 9.
- Train only encoder layers 10 and 11 (~14M parameters) plus the token classification head.
- **Differential Learning Rates:**
  - Head (`classifier`): `1e-3`
  - Trainable top layers (10-11): `2e-5`


In [ ]:
model_partial = AutoModelForTokenClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(device)

# Freeze embeddings and layers 0 through 9
for param in model_partial.bert.embeddings.parameters():
    param.requires_grad = False

for layer in model_partial.bert.encoder.layer[:10]:
    for param in layer.parameters():
        param.requires_grad = False

# Separate parameters for differential learning rates
head_params_partial = []
top_layer_params_partial = []

for name, param in model_partial.named_parameters():
    if not param.requires_grad:
        continue
    if "classifier" in name:
        head_params_partial.append(param)
    else:
        top_layer_params_partial.append(param)

optimizer_partial = AdamW([
    {"params": top_layer_params_partial, "lr": 2e-5, "weight_decay": 0.01},
    {"params": head_params_partial, "lr": 1e-3, "weight_decay": 0.01}
])

trainable_params_a = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
total_params_a = sum(p.numel() for p in model_partial.parameters())
print(f"[Experiment A - Partial FT] Trainable: {trainable_params_a:,} / {total_params_a:,} ({trainable_params_a/total_params_a*100:.2f}%)")

epochs = 3
total_steps_a = len(train_loader) * epochs
scheduler_partial = get_linear_schedule_with_warmup(optimizer_partial, num_warmup_steps=int(total_steps_a*0.1), num_training_steps=total_steps_a)

history_a = {"train_loss": [], "val_loss": [], "precision": [], "recall": [], "f1": [], "epoch_times": []}

print("\n--- Starting Training: Experiment A (Partial Fine-Tuning) ---")
for epoch in range(epochs):
    t0 = time.time()
    model_partial.train()
    total_train_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_partial.zero_grad()
        outputs = model_partial(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(model_partial.parameters(), 1.0)
        optimizer_partial.step()
        scheduler_partial.step()

        total_train_loss += loss.item() * input_ids.size(0)

    epoch_time = time.time() - t0
    train_loss = total_train_loss / len(train_loader.dataset)
    val_loss, val_prec, val_rec, val_f1, _ = evaluate_ner(model_partial, val_loader, device)

    history_a["train_loss"].append(train_loss)
    history_a["val_loss"].append(val_loss)
    history_a["precision"].append(val_prec)
    history_a["recall"].append(val_rec)
    history_a["f1"].append(val_f1)
    history_a["epoch_times"].append(epoch_time)

    print(f"Epoch {epoch+1}/{epochs} | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | F1: {val_f1:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f}")


### 7. Experiment B: Full Fine-Tuning (All Layers + Head)
**Concept:**
- All 12 layers of BERT + embeddings + classification head are updated.
- **Differential Learning Rates:**
  - Head (`classifier`): `1e-3`
  - Full BERT backbone: `2e-5`


In [ ]:
model_full = AutoModelForTokenClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(device)

head_params_full = []
backbone_params_full = []

for name, param in model_full.named_parameters():
    if "classifier" in name:
        head_params_full.append(param)
    else:
        backbone_params_full.append(param)

optimizer_full = AdamW([
    {"params": backbone_params_full, "lr": 2e-5, "weight_decay": 0.01},
    {"params": head_params_full, "lr": 1e-3, "weight_decay": 0.01}
])

trainable_params_b = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
print(f"[Experiment B - Full FT] Trainable: {trainable_params_b:,} (100% trainable)")

total_steps_b = len(train_loader) * epochs
scheduler_full = get_linear_schedule_with_warmup(optimizer_full, num_warmup_steps=int(total_steps_b*0.1), num_training_steps=total_steps_b)

history_b = {"train_loss": [], "val_loss": [], "precision": [], "recall": [], "f1": [], "epoch_times": []}

print("\n--- Starting Training: Experiment B (Full Fine-Tuning) ---")
for epoch in range(epochs):
    t0 = time.time()
    model_full.train()
    total_train_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_full.zero_grad()
        outputs = model_full(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(model_full.parameters(), 1.0)
        optimizer_full.step()
        scheduler_full.step()

        total_train_loss += loss.item() * input_ids.size(0)

    epoch_time = time.time() - t0
    train_loss = total_train_loss / len(train_loader.dataset)
    val_loss, val_prec, val_rec, val_f1, _ = evaluate_ner(model_full, val_loader, device)

    history_b["train_loss"].append(train_loss)
    history_b["val_loss"].append(val_loss)
    history_b["precision"].append(val_prec)
    history_b["recall"].append(val_rec)
    history_b["f1"].append(val_f1)
    history_b["epoch_times"].append(epoch_time)

    print(f"Epoch {epoch+1}/{epochs} | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | F1: {val_f1:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f}")


### 8. Evaluation on Test Set & Comparative Analysis
We evaluate both models on the held-out test split of CoNLL-2003 and build our comparison matrix.


In [ ]:
# Evaluate on final test split
_, test_prec_a, test_rec_a, test_f1_a, test_acc_a = evaluate_ner(model_partial, test_loader, device)
_, test_prec_b, test_rec_b, test_f1_b, test_acc_b = evaluate_ner(model_full, test_loader, device)

comparison_df = pd.DataFrame({
    "Method": ["Partial Fine-Tuning (Top 2 Layers)", "Full Fine-Tuning (Differential LR)"],
    "Trainable Params": [f"{trainable_params_a:,}", f"{trainable_params_b:,}"],
    "Total Train Time (s)": [f"{sum(history_a['epoch_times']):.2f}", f"{sum(history_b['epoch_times']):.2f}"],
    "Avg Sec/Epoch": [f"{np.mean(history_a['epoch_times']):.2f}", f"{np.mean(history_b['epoch_times']):.2f}"],
    "Test Precision": [f"{test_prec_a:.4f}", f"{test_prec_b:.4f}"],
    "Test Recall": [f"{test_rec_a:.4f}", f"{test_rec_b:.4f}"],
    "Test F1 (seqeval)": [f"{test_f1_a:.4f}", f"{test_f1_b:.4f}"],
    "Status": ["Rejected Alternative", "Delivered Model"]
})

print("=== CONLL-2003 NER COMPARISON SUMMARY ===")
display(comparison_df)

# Visualizations
epochs_range = range(1, epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss Curves
axes[0].plot(epochs_range, history_a["train_loss"], 'o--', label="Train Loss (Partial FT)", color="purple")
axes[0].plot(epochs_range, history_a["val_loss"], 'o-', label="Val Loss (Partial FT)", color="orchid")
axes[0].plot(epochs_range, history_b["train_loss"], 's--', label="Train Loss (Full FT)", color="teal")
axes[0].plot(epochs_range, history_b["val_loss"], 's-', label="Val Loss (Full FT)", color="darkcyan")
axes[0].set_title("Training & Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# F1 Curves
axes[1].plot(epochs_range, history_a["f1"], 'o-', label="Val F1 (Partial FT)", color="purple")
axes[1].plot(epochs_range, history_b["f1"], 's-', label="Val F1 (Full FT)", color="teal")
axes[1].plot(epochs_range, history_a["precision"], 'o--', label="Val Prec (Partial FT)", color="orchid")
axes[1].plot(epochs_range, history_b["precision"], 's--', label="Val Prec (Full FT)", color="darkcyan")
axes[1].set_title("Validation F1 & Precision (seqeval)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("task2_ner_comparison.png", dpi=300)
plt.show()


### 9. Export & Publication to Hugging Face Hub
We save the delivered model (`model_full`), package the tokenizer, and generate the **Model Card** (`README.md`).

> If private, grant access to user `Dexterg83` as required.


In [ ]:
from huggingface_hub import login, HfApi

save_dir = "./delivered_model_ner"
model_full.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Delivered model saved to {save_dir}")

# Construct Model Card
model_card_content = f"""---
language: en
license: mit
tags:
- token-classification
- bert
- ner
- conll2003
datasets:
- conll2003
metrics:
- seqeval
- f1
- precision
- recall
---

# BERT-base-cased for Named Entity Recognition (CoNLL-2003) - Delivered Model

## Model Description
This model is a fine-tuned `bert-base-cased` checkpoint adapted for token classification / Named Entity Recognition on the standard **CoNLL-2003** dataset (PER, ORG, LOC, MISC, O).

## Subword Alignment & Preprocessing
- Tokenizer: `bert-base-cased`
- First subwords receive the designated NER entity tag.
- Subsequent subword pieces, `[CLS]`, `[SEP]`, and padding tokens are masked with label `-100`, ignoring them during Cross-Entropy Loss computation.

## Adaptation Comparison: Delivered vs Rejected Alternative
- **Delivered Model:** Full Fine-Tuning (All 12 layers + Head with differential learning rates: Head `1e-3`, Backbone `2e-5`).
- **Rejected Alternative:** Partial Fine-Tuning (Frozen layers 0-9; only top 2 layers + Head trained).

### Empirical Performance Comparison:
- **Full Fine-Tuning Test F1:** {test_f1_b:.4f} | **Precision:** {test_prec_b:.4f} | **Recall:** {test_rec_b:.4f}
- **Partial Fine-Tuning Test F1:** {test_f1_a:.4f} | **Precision:** {test_prec_a:.4f} | **Recall:** {test_rec_a:.4f}
- **Technical Justification:** Full fine-tuning allows token-level self-attention weights across middle and lower layers to coordinate entity boundary syntax and contextual semantic disambiguation, yielding a substantial gain in entity-level F1.

## Training Configuration
- **Base Checkpoint:** `bert-base-cased` (110M params)
- **Differential Optimizer:** AdamW
  - Backbone LR: 2e-5
  - Head LR: 1e-3
- **Batch Size:** {batch_size}
- **Epochs:** {epochs}
- **Seed:** 42

## Access Permissions
If private, collaborator access has been granted to user `Dexterg83` per course assignment instructions.
"""

with open(os.path.join(save_dir, "README.md"), "w") as f:
    f.write(model_card_content)
print("Model card README.md created.")

# To push to Hugging Face Hub in Colab:
# login(token="YOUR_HF_TOKEN")
# hf_repo_name = "YOUR_USERNAME/bert-base-cased-conll2003-ner"
# model_full.push_to_hub(hf_repo_name, private=False)
# tokenizer.push_to_hub(hf_repo_name, private=False)
print("Ready for Hugging Face Hub upload. Uncomment the lines above with your token and repo name.")
